In [28]:
import numpy as np
from pathlib import Path
from ase.build import bulk
from ase.io import write
from ase.calculators.espresso import Espresso, EspressoProfile
from ase.visualize import view
from ase.visualize import view as viewer
import sys

helper_path = Path("/Users/famkepotze/Desktop/3S/")
sys.path.insert(0, str(helper_path))
from helper import supercell_size


In [33]:
#variables   
# ── Build NV structure ────────────────────────────────────────────
atoms = bulk('C', 'diamond', cubic=True) * supercell_size[3]

center = atoms.get_cell().sum(axis=0) / 2
distances = np.linalg.norm(atoms.get_positions() - center, axis=1)
center_index = np.argmin(distances)

dists = atoms.get_distances(center_index, range(len(atoms)), mic=True)
dists[center_index] = 1e6
nn_index = np.argmin(dists)

del atoms[center_index]
if nn_index > center_index:
    nn_index -= 1
atoms[nn_index].symbol = 'N'

# NV center has a spin — set spin-polarized with 2 unpaired electrons
atoms.set_initial_magnetic_moments(
    [2.0 if a.symbol == 'N' else 0.0 for a in atoms]
)

save_path = Path(f"/Users/famkepotze/Desktop/3S/QuantumEspresso/results/raw_structures/NV_{supercell_size[3][0]}x{supercell_size[3][1]}x{supercell_size[3][2]}.xyz")
save_path.parent.mkdir(parents=True, exist_ok=True)
write(save_path, atoms)

view(atoms, viewer='x3d')